# Auto ARIMA Example

Uses `pmdarima.auto_arima` to automatically select the best (p, d, q) × (P, D, Q, m) order for a SARIMA model based on information criteria (AIC/BIC).

We'll use the classic **Airline Passengers** dataset (monthly, 1949–1960).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import pmdarima as pm

import autoarimahelper as aah

warnings.filterwarnings("ignore")

## 1. Load and prepare data

In [ ]:
# Use pmdarima's built-in airline passengers dataset
y = pm.datasets.load_airpassengers()

# Create a datetime index for plotting
dates = pd.date_range(start="1949-01", periods=len(y), freq="MS")
ts = pd.Series(y, index=dates, name="Passengers")

# Train/test split — hold out last 24 months
n_test = 24
train, test = aah.train_test_split(ts, n_test=n_test)

print(f"Train: {train.index[0].date()} to {train.index[-1].date()}  (n={len(train)})")
print(f"Test:  {test.index[0].date()} to {test.index[-1].date()}  (n={len(test)})")

In [ ]:
ts.plot(figsize=(10, 4), title="Airline Passengers (1949–1960)")
plt.ylabel("Passengers")
plt.tight_layout()
plt.show()

## 2. Fit auto_arima (seasonal)

Key parameters:
- `seasonal=True, m=12` — monthly seasonality
- `stepwise=True` — faster search (default)
- `trace=True` — prints each model tried and its AIC
- `d` and `D` — set to `None` to let auto_arima determine differencing via tests (KPSS for `d`, OCSB for `D`)

In [ ]:
result = aah.fit_auto_arima(
    train,
    seasonal=True,
    m=12,
    d=None,
    D=None,
    max_p=5,
    max_q=5,
    max_P=2,
    max_Q=2,
    stepwise=True,
    trace=True,
    information_criterion="aic",
)

print(f"\nBest model: {result}")
print(f"Order: {result.order} x {result.seasonal_order}")

## 3. Model summary

In [ ]:
result.model.summary()

## 4. Diagnostics

In [ ]:
result.model.plot_diagnostics(figsize=(10, 8))
plt.tight_layout()
plt.show()

## 5. Forecast and evaluate on test set

In [ ]:
# Forecast
fc = aah.forecast(result, n_periods=n_test, return_conf_int=True)

# Evaluate
metrics = aah.evaluate_forecast(test.values, fc.values)
print(f"Test RMSE: {metrics.rmse:.2f}")
print(f"Test MAE:  {metrics.mae:.2f}")

# Plot
ax = aah.plot_forecast(
    train, test, fc,
    fc_index=test.index,
    title=f"Auto ARIMA {result.order}x{result.seasonal_order}  —  RMSE={metrics.rmse:.2f}",
)
plt.tight_layout()
plt.show()

## 6. Update model with new observations (online learning)

`pmdarima` supports updating the model with new data without refitting from scratch.

In [ ]:
# Update with first 12 test observations, then forecast the remaining 12
aah.update_model(result, test[:12])

fc2 = aah.forecast(result, n_periods=12, return_conf_int=True)
metrics2 = aah.evaluate_forecast(test[12:].values, fc2.values)
print(f"Updated model RMSE (last 12 months): {metrics2.rmse:.2f}")
print(f"Updated model MAE  (last 12 months): {metrics2.mae:.2f}")